# AnalyzeTCSPC
Step-by-step TCSPC-FLIM analysis: IRF correction, per-pixel mean-arrival-time lifetime map (`tav`), false-colour image, decay fitting, phasor.

Run cells **in order**. Edit the **Parameters** cell for each experiment.

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'   # prevent OpenMP conflict between MKL and torch on Windows

import torch   # must be imported before numpy/scipy to avoid OpenMP runtime clash

import sys
import csv
import glob
import re
import warnings
import importlib
import tkinter as tk
from tkinter import filedialog
from collections import defaultdict

from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
import tifffile
import pandas as pd
from scipy.ndimage import median_filter, gaussian_filter
from skimage.restoration import (denoise_bilateral,
                                  denoise_nl_means as _nlm_fn,
                                  rolling_ball as _rolling_ball_fn,
                                  estimate_sigma as _esig_fn)
from imageio import imwrite
from cellpose import models as _cp_models

sys.path.insert(0, 'X:/FLIM_2026')
import FastFLIM_mod_AT0 as _ffm_module
from FastFLIM_mod_AT0 import FastFLIM_mod_AT0
from flim_helpers import uplowthresh, Fit_decay, Calc_Phasor_Mat, phasor_ref_correction, read_fbs_metadata

print(f'torch {torch.__version__}  CUDA {torch.version.cuda}  GPU available: {torch.cuda.is_available()}')

## Helper functions
`uplowthresh` finds upper/lower display thresholds from the pixel histogram.
`Fit_decay` fits a multi-exponential model to the spatially-averaged decay.
`Calc_Phasor_Mat` computes phasor components and the density matrix.

## Step 1: Parameters & data loading
Edit the analysis parameters, then run the cell — file dialogs will open (IRF first, then data folder).
Set `dim_data` / `dim_irf` to match your TIFF axis order if needed.

| Variable | Shape | Description |
|----------|-------|-------------|
| `DataTCSPC` | `(ny, nx, nbin)` | TCSPC photon counts after reorder |
| `IRF` | `(nbin,)` | Instrument Response Function (summed to 1-D) |
| `sumTCSPC` | `(ny, nx)` | Total photons per pixel |
| `maskPMT` | `(ny, nx)` | 1 = valid pixel (counts > threshold), NaN = masked |

In [ ]:
# ── analysis parameters (edit per experiment) ─────────────────────────────────
Tcycle    = 12.5      # fallback if no .fbs.xml found — overridden automatically
threshold = 3
limits    = np.array([np.nan, np.nan])
szPhs     = 0.005
szBin     = 3

thresh_norm    = 0.55
nlm_h          = None
nlm_patch_size = 5
nlm_patch_dist = 6
nlm_fast       = True
rb_radius      = 30

Data_path_main = os.path.normpath(r'K:\Imaging\Peredox_Awake\ISS\20260505_Peredox_awake_FLIM\XYZ_272_PostStroke_2')

# ── TIFF dimension order ───────────────────────────────────────────────────────
dim_data = 'TYX'
dim_irf  = 'TYX'

# ── file / folder pickers (uncomment for interactive selection) ───────────────
def _pick_file(title):
    root = tk.Tk(); root.withdraw(); root.attributes('-topmost', True)
    p = filedialog.askopenfilename(
        title=title, filetypes=[('TIFF files', '*.tif *.tiff'), ('All files', '*.*')])
    root.destroy()
    return p or None

def _pick_folder(title, initialdir=Data_path_main):
    root = tk.Tk(); root.withdraw(); root.attributes('-topmost', True)
    p = filedialog.askdirectory(title=title, initialdir=initialdir)
    root.destroy()
    return p or None

# ── paths — swap comments to switch between hardcoded and interactive ─────────
irf_path    = r'X:\FLIM_2026\test_data\IRF\IRF_840nm_Intensity.tif'
# irf_path    = _pick_file('Select IRF TIFF')
data_folder = _pick_folder('Select folder containing data TIFFs (z-stack)', initialdir=r'X:\FLIM_2026\test_data')

if not data_folder:
    raise RuntimeError('No data folder selected.')

# ── FName derived from folder name ────────────────────────────────────────────
FName = os.path.basename(data_folder)
_ts       = datetime.now().strftime('%Y%m%d_%H%M')
newFolder = os.path.join(data_folder, f'{_ts}_FLIM_Analysis')
os.makedirs(newFolder, exist_ok=True)
print(f'  IRF:         {irf_path}')
print(f'  Data folder: {data_folder}')
print(f'  FName:       {FName}')

# ── natural sort ──────────────────────────────────────────────────────────────
def _natural_key(s):
    return [int(c) if c.isdigit() else c.lower() for c in re.split(r'(\d+)', s)]

# ── XML search helper ─────────────────────────────────────────────────────────
def _find_xml(tiff_path):
    base = re.sub(r'_Intensity.*$', '', os.path.splitext(os.path.basename(tiff_path))[0])
    for search_dir in [os.path.dirname(os.path.dirname(tiff_path)), os.path.dirname(tiff_path)]:
        hits = sorted(glob.glob(os.path.join(search_dir, '*.fbs.xml')))
        if hits:
            return max(hits, key=lambda p: len(os.path.commonprefix([base, os.path.basename(p)])))
    return None

# ── axis-reorder helper ───────────────────────────────────────────────────────
def _to_yxt(arr, order):
    order = list(order.upper())
    extra = [i for i, c in enumerate(order) if c not in ('T', 'Y', 'X')]
    if extra:
        arr   = arr.sum(axis=tuple(sorted(extra)))
        order = [c for c in order if c in ('T', 'Y', 'X')]
    y, x, t = order.index('Y'), order.index('X'), order.index('T')
    return np.transpose(arr, [y, x, t])

# ── enumerate z-planes ────────────────────────────────────────────────────────
data_paths = sorted(
    glob.glob(os.path.join(data_folder, '*.tif')) +
    glob.glob(os.path.join(data_folder, '*.tiff')),
    key=_natural_key)
n_planes = len(data_paths)
mid_idx  = n_planes // 2
print(f'\nFound {n_planes} planes in:  {data_folder}')
print(f'Middle plane  z={mid_idx}:  {os.path.basename(data_paths[mid_idx])}')

# ── load IRF metadata ─────────────────────────────────────────────────────────
meta_irf = None
xml_irf  = _find_xml(irf_path)
if xml_irf:
    meta_irf = read_fbs_metadata(xml_irf)
    print(f'\nIRF metadata:  {os.path.basename(xml_irf)}')
    print(f'  Tcycle={meta_irf["Tcycle"]:.4f} ns  nbin={meta_irf["nbin"]}  '
          f'gain={meta_irf["detector_gain"]}%  dwell={meta_irf["dwell_time_ms"]} ms/px')
else:
    print('\n  No .fbs.xml found for IRF')

# ── load metadata from middle plane ──────────────────────────────────────────
meta     = None
xml_data = _find_xml(data_paths[mid_idx])
if xml_data:
    meta   = read_fbs_metadata(xml_data)
    Tcycle = meta['Tcycle']
    print(f'\nData metadata: {os.path.basename(xml_data)}')
    print(f'  Tcycle={Tcycle:.4f} ns  nbin={meta["nbin"]}  '
          f'pixel={meta["pixel_size_um"]:.4f} µm  '
          f'gain={meta["detector_gain"]}%  dwell={meta["dwell_time_ms"]} ms/px  '
          f'excitation={meta["excitation_nm"]} nm')
else:
    warnings.warn('No .fbs.xml found for data — using manual Tcycle.', RuntimeWarning, stacklevel=2)
    print(f'\n  No XML found — using manual Tcycle = {Tcycle} ns')

# ── Tcycle cross-check ────────────────────────────────────────────────────────
if meta_irf and meta:
    _dt = abs(meta_irf['Tcycle'] - meta['Tcycle'])
    if _dt > 0.01:
        warnings.warn(
            f'Tcycle mismatch: IRF={meta_irf["Tcycle"]:.4f} ns, '
            f'data={meta["Tcycle"]:.4f} ns (diff={_dt:.4f} ns). Using data Tcycle.',
            RuntimeWarning, stacklevel=2)
        print(f'WARNING: Tcycle mismatch — IRF={meta_irf["Tcycle"]:.4f}, data={meta["Tcycle"]:.4f} ns')
    else:
        print(f'\nTcycle cross-check OK  ({meta_irf["Tcycle"]:.4f} vs {meta["Tcycle"]:.4f} ns,  '
              f'diff={_dt:.5f} ns)')
elif meta_irf and not meta:
    Tcycle = meta_irf['Tcycle']
    print(f'Using Tcycle from IRF XML: {Tcycle:.4f} ns')

_gain  = f"Gain {meta['detector_gain']}%"        if meta and meta['detector_gain']  is not None else ''
_dwell = f"Dwell {meta['dwell_time_ms']} ms/px"   if meta and meta['dwell_time_ms'] is not None else ''
meta_str = '  ·  '.join(s for s in [_gain, _dwell] if s)

# ── load IRF (1-D) ────────────────────────────────────────────────────────────
IRF_raw = tifffile.imread(irf_path).astype(float)
IRF = (_to_yxt(IRF_raw, dim_irf).sum(axis=(0, 1))
       if any(c in dim_irf.upper() for c in ('Y', 'X')) else IRF_raw.ravel())

# ── load middle plane as reference DataTCSPC ─────────────────────────────────
data_raw  = tifffile.imread(data_paths[mid_idx]).astype(float)
DataTCSPC = _to_yxt(data_raw, dim_data)
print(f'\nMiddle-plane DataTCSPC shape: {DataTCSPC.shape}  (ny, nx, nbin)')

# ── nbin check ────────────────────────────────────────────────────────────────
nbin_data = DataTCSPC.shape[2]
for _label, _m in [('data XML', meta), ('IRF XML', meta_irf)]:
    if _m and _m['nbin'] is not None and nbin_data != _m['nbin']:
        warnings.warn(
            f'nbin mismatch ({_label}): data has {nbin_data} bins, XML says {_m["nbin"]}.',
            RuntimeWarning, stacklevel=2)
print(f'nbin = {nbin_data}  ✓')

sumTCSPC = DataTCSPC.sum(axis=2)
maskPMT  = (sumTCSPC >= threshold).astype(float)
maskPMT[maskPMT == 0] = np.nan

_flat  = sumTCSPC.ravel(); _flat = _flat[_flat > 0]
lo_pct = np.percentile(_flat, 1)
hi_pct = np.percentile(_flat, 99.9)

_nbin_d  = DataTCSPC.shape[2]
_tau_d   = np.arange(_nbin_d) * (Tcycle / _nbin_d)
_decay   = DataTCSPC.sum(axis=(0, 1)).astype(float)
_irf_d   = IRF[:_nbin_d] / IRF[:_nbin_d].max()
_decay_n = _decay / _decay.max()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
im = axes[0].imshow(sumTCSPC, cmap='inferno', aspect='equal', origin='upper',
                    vmin=lo_pct, vmax=hi_pct)
axes[0].set_title(f'Intensity  [{FName}]  z={mid_idx}/{n_planes-1}', fontsize=12)
axes[0].set_xlabel('x / px'); axes[0].set_ylabel('y / px')
plt.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04).set_label('photon count')
axes[1].semilogy(_tau_d, _decay_n, color='steelblue', linewidth=1.2, label='Data (mid)')
axes[1].semilogy(_tau_d, _irf_d,   color='k', linewidth=0.8, linestyle='--', label='IRF')
axes[1].set_xlabel('time / ns'); axes[1].set_ylabel('intensity (norm.)')
axes[1].set_title('Spatially-averaged decay  (middle plane)', fontsize=12)
axes[1].set_xlim(0, Tcycle); axes[1].set_ylim(5e-3, 1.3)
axes[1].legend()
_suptitle = FName + (f'  —  {meta_str}' if meta_str else '')
plt.suptitle(_suptitle, fontsize=10, y=1.01)
plt.tight_layout()
plt.show()

## Step 2: 3D pre-processing & Cellpose segmentation
Auto-detects whether pre-computed results already exist in `newFolder`:
- **Results found** → reloads `vol_norm_3d`, `mask3d`, `cp_masks` instantly.
- **No results** → runs 3D NLM denoising + rolling-ball BG subtraction → Cellpose 3D segmentation, then saves all outputs.

To force a re-run, delete (or rename) the `*_sumTCSPC_3Dfilt_norm.tif` / `*_mask3d.tif` files in `newFolder`.

In [ ]:
# ── Cellpose / pre-processing parameters ─────────────────────────────────────
cp_diameter        = 50     # expected cell diameter in xy (pixels)
cp_anisotropy      = 3.0     # z_step_um / xy_pixel_um  (update per acquisition)
cp_model_type      = 'cpsam' # Cellpose 4 unified model; 'cyto3'/'nuclei' are aliases
cp_flow_threshold  = 0.1     # raise to accept more cells
cp_cellprob_thresh = 0.9     # lower = more cells detected
cp_gpu             = True

# ── check for pre-computed outputs ───────────────────────────────────────────
_filt_path = os.path.join(newFolder, f'{FName}_sumTCSPC_3Dfilt_norm.tif')
_mask_path = os.path.join(newFolder, f'{FName}_mask3d.tif')
_cp_path   = os.path.join(newFolder, f'{FName}_cellpose_masks.tif')
_have_pre  = os.path.exists(_filt_path) and os.path.exists(_mask_path)
_rerun_cp  = False   # set to False to skip Cellpose re-run when pre-processing is redone   
_have_cp   = os.path.exists(_cp_path)

if _have_pre and not _rerun_cp:
    # ── reload pre-processed volume ───────────────────────────────────────
    vol_norm_3d = tifffile.imread(_filt_path).astype(np.float32)
    mask3d      = tifffile.imread(_mask_path).astype(bool)
    print(f'Reloaded  vol_norm_3d  {vol_norm_3d.shape}  <- {os.path.basename(_filt_path)}')
    print(f'Reloaded  mask3d       {mask3d.shape}  valid: {mask3d.sum():,} voxels')
else:
    # ── build 3D sumTCSPC volume ──────────────────────────────────────────
    print(f'Loading {n_planes} planes...', end=' ', flush=True)
    vol_sum3d = np.stack([
        _to_yxt(tifffile.imread(p).astype(float), dim_data).sum(axis=2)
        for p in data_paths
    ], axis=0).astype(np.float32)
    print(f'done.  Shape: {vol_sum3d.shape}')

    # ── pre-normalise → 3D NLM ───────────────────────────────────────────
    _vlo = np.percentile(vol_sum3d, 1);  _vhi = np.percentile(vol_sum3d, 99)
    _vol01 = np.clip((vol_sum3d - _vlo) / (_vhi - _vlo), 0, 1).astype(np.float32)
    _sigma3d = float(_esig_fn(_vol01, average_sigmas=True, channel_axis=None))
    _h3d     = 0.8 * _sigma3d if nlm_h is None else nlm_h
    print(f'Noise sigma ~ {_sigma3d:.5f}   h = {_h3d:.5f}')
    print('Running 3D NLM (CPU)...', end=' ', flush=True)
    _vol_nlm = _nlm_fn(_vol01, h=_h3d, patch_size=nlm_patch_size,
                       patch_distance=nlm_patch_dist, fast_mode=nlm_fast,
                       channel_axis=None).astype(np.float32)
    print('done.')

    # ── rolling ball BG subtraction ───────────────────────────────────────
    print('Rolling ball...', end=' ', flush=True)
    _vol_rb = np.empty_like(_vol_nlm)
    for _iz in range(_vol_nlm.shape[0]):
        _bg = _rolling_ball_fn(_vol_nlm[_iz], radius=rb_radius, workers=-1)
        _vol_rb[_iz] = np.clip(_vol_nlm[_iz] - _bg, 0, None)
    print('done.')

    # ── final normalise → threshold mask ─────────────────────────────────
    _vlo3d, _vhi3d = np.percentile(_vol_rb, 1), np.percentile(_vol_rb, 99)
    vol_norm_3d = np.clip((_vol_rb - _vlo3d) / (_vhi3d - _vlo3d), 0, 1).astype(np.float32)
    mask3d = vol_norm_3d >= thresh_norm

    # ── save ─────────────────────────────────────────────────────────────
    tifffile.imwrite(_filt_path, vol_norm_3d)
    tifffile.imwrite(_mask_path, mask3d.astype(np.uint8))
    print(f'Saved: {_filt_path}')
    print(f'Valid voxels: {mask3d.sum():,} / {mask3d.size:,}  ({100*mask3d.mean():.1f}%)  thresh_norm={thresh_norm}')
    _have_cp = False   # force Cellpose re-run when pre-processing was redone

if _have_cp:
    # ── reload Cellpose masks ─────────────────────────────────────────────
    cp_masks = tifffile.imread(_cp_path)
    n_cells  = int(cp_masks.max())
    print(f'Reloaded  cp_masks  {cp_masks.shape}  {n_cells} cells')
else:
    # ── run Cellpose 3D ───────────────────────────────────────────────────
    print(f'Running Cellpose 3D  model=cpsam  diam={cp_diameter}px  '
          f'aniso={cp_anisotropy}  gpu={cp_gpu}...', flush=True)
    _cp_model = _cp_models.CellposeModel(gpu=cp_gpu)
    cp_masks, cp_flows, cp_styles = _cp_model.eval(
        vol_norm_3d,
        diameter=cp_diameter, z_axis=0, do_3D=True,
        anisotropy=cp_anisotropy,
        flow_threshold=cp_flow_threshold,
        cellprob_threshold=cp_cellprob_thresh,
        min_size=500,
    )
    n_cells = int(cp_masks.max())
    print(f'Done.  {n_cells} cells found.')
    tifffile.imwrite(_cp_path, cp_masks.astype(np.uint16))
    print(f'Saved: {_cp_path}')

# ── QC: middle plane ──────────────────────────────────────────────────────────
_fig_pre, _axes_pre = plt.subplots(1, 3, figsize=(18, 5), facecolor='black')
for _a in _axes_pre:
    _a.set_facecolor('black')
_n_lab   = max(n_cells, 1)
_cmap_cp = plt.colormaps.get_cmap('tab20').resampled(_n_lab)
_panels  = [
    (vol_norm_3d[mid_idx], 'inferno',  f'NLM + rolling ball  (z={mid_idx})'),
    (mask3d[mid_idx].astype(float), 'inferno', f'mask  (thresh_norm={thresh_norm})'),
    (cp_masks[mid_idx],    _cmap_cp,  f'Cellpose  (z={mid_idx})  {n_cells} cells'),
]
for _a, (_img, _cm, _ttl) in zip(_axes_pre, _panels):
    _kw = dict(vmin=0, vmax=_n_lab, interpolation='nearest') if 'Cellpose' in _ttl else {}
    _a.imshow(_img, cmap=_cm, aspect='equal', origin='upper', **_kw)
    _a.set_title(_ttl, color='white', fontsize=10)
    _a.tick_params(colors='white')
    for _sp in _a.spines.values():
        _sp.set_edgecolor('white')
plt.suptitle(f'{FName}  —  Step 2: pre-processing & segmentation  (z={mid_idx}/{n_planes-1})',
             color='white', fontsize=9)
plt.tight_layout()
plt.show()

## Step 3: Mask preparation (middle plane)
Extracts the middle-plane slice of `vol_norm_3d` and thresholds at `thresh_norm` to produce `maskTCSPC` (float, 0/1) and `maskFilt_bool` (boolean).

In [ ]:
# ── masks from 3D pre-processed volume (middle plane) ────────────────────────
maskTCSPC     = mask3d[mid_idx].astype(float)
maskFilt_bool = mask3d[mid_idx]

# ── save middle-plane intensity maps ─────────────────────────────────────────
imwrite(os.path.join(newFolder, f'{FName}_sumTCSPC.tif'), sumTCSPC.astype(np.float32))
imwrite(os.path.join(newFolder, f'{FName}_sumTCSPC_filt_norm.tif'),
        vol_norm_3d[mid_idx].astype(np.float32))

print(f'maskTCSPC valid pixels: {maskFilt_bool.sum():.0f}  (thresh_norm={thresh_norm})')

## Step 4: IRF preprocessing
1. Sum over spatial dims to get a 1-D IRF trace.
2. Subtract median of last 25 bins (background) and clip negatives.

In [ ]:
IRF2 = IRF.sum(axis=tuple(range(IRF.ndim - 1))) if IRF.ndim > 1 else IRF.copy()
IRF2 = IRF2.ravel().astype(float)
Ix   = len(IRF2)

# IRF2 = np.roll(IRF2, 2)           # 2-bin right-shift -- legacy from old instrument, not needed
IRF2 = IRF2 - np.median(IRF2[-25:]) # subtract background (median of last 25 bins)
IRF2 = IRF2 + np.abs(IRF2.min())    # clip negatives to zero

print(f'IRF bins (Ix): {Ix}')
print(f'IRF peak at bin {np.argmax(IRF2)},  max = {IRF2.max():.1f}')


## Step 5: Peak realignment
Circularly shifts `DataTCSPC` and `IRF2` so both peaks land at 8 % of `nbin` (the same target frame). This single global alignment replaces the old per-session manual shift and ensures `F - H` is unbiased.

In [ ]:
## Step 4b: Peak realignment
# Circularly shifts DataTCSPC and IRF2 so both peaks land at 8% of nbin.

# ── bin count check ───────────────────────────────────────────────────────────
nbin_data = DataTCSPC.shape[2]
if nbin_data != Ix:
    warnings.warn(
        f'Bin count mismatch: DataTCSPC has {nbin_data} bins, IRF has {Ix} bins. '
        'Check dim_data / dim_irf — bin alignment (code-06) may need re-enabling.',
        RuntimeWarning, stacklevel=2)
nbin = nbin_data
dt   = Tcycle / nbin
tau  = np.arange(1, nbin + 1) * dt
print(f'nbin = {nbin},  dt = {dt:.4f} ns')

# ── target frame: 8% of nbin ─────────────────────────────────────────────────
target = int(round(0.08 * nbin))
print(f'Target peak frame: {target} / {nbin}  ({100 * target / nbin:.1f}%)')

# ── pixel mask ────────────────────────────────────────────────────────────────
maskPixel    = mask3d[mid_idx]
decay_masked = DataTCSPC[maskPixel].sum(axis=0).astype(float)

# ── shift DataTCSPC so fluorescence peak → target frame ──────────────────────
peak_data  = int(np.argmax(decay_masked))
shift_data = target - peak_data
DataTCSPC  = np.roll(DataTCSPC, shift_data, axis=2)
print(f'Data  peak was frame {peak_data:3d},  shifted by {shift_data:+d}')

# ── shift IRF2 so its peak → target frame ────────────────────────────────────
peak_irf  = int(np.argmax(IRF2))
shift_irf = target - peak_irf
IRF2      = np.roll(IRF2, shift_irf)
print(f'IRF   peak was frame {peak_irf:3d},  shifted by {shift_irf:+d}')

# ── sanity-check plot ─────────────────────────────────────────────────────────
_decay_aligned = DataTCSPC[maskPixel].sum(axis=0).astype(float)
_decay_n       = _decay_aligned / _decay_aligned.max()
_irf_n         = IRF2 / IRF2.max()

fig_align, ax_align = plt.subplots(figsize=(7, 4))
ax_align.semilogy(tau, _irf_n, '--k', linewidth=0.8, label='IRF')
ax_align.semilogy(tau, _decay_n, color='steelblue', linewidth=1.2,
                  label=f'Data  ({maskPixel.sum()} px)')
ax_align.axvline(tau[target], color='orange', linewidth=1, linestyle=':',
                 label=f'target  frame {target}')
ax_align.set_xlabel('time / ns')
ax_align.set_ylabel('intensity (norm.)')
ax_align.set_title(f'After realignment  [{FName}]' + (f'\n{meta_str}' if meta_str else ''),
                   fontsize=10)
ax_align.set_xlim(0, Tcycle)
ax_align.set_ylim(5e-3, 1.3)
ax_align.legend()
fig_align.tight_layout()
plt.show()

## Step 6: Masked decay & time axis
Builds the time axis `tau` and computes the spatially-masked average decay `tmp1` used for alignment and fitting.

In [ ]:
# DataTCSPC = np.roll(DataTCSPC, 2, axis=2)   # 2-bin shift -- not needed for new instrument

dt  = Tcycle / nbin
tau = np.arange(1, nbin + 1) * dt               # time axis (ns)

# masked average decay (pixels >= threshold) -- consistent with maskPixel
# convert maskTCSPC to boolean for indexing, but keep original maskPixel for stats
maskTCSPCbool = maskTCSPC.astype(bool)
tmp1 = DataTCSPC[maskPixel].sum(axis=0).astype(float)

tmp = tmp1 / tmp1.max()
irf = IRF2 / IRF2.max()

print(f'dt = {dt:.4f} ns,  tau: {tau[0]:.3f} to {tau[-1]:.3f} ns')
print(f'Total photons in masked area: {tmp1.sum():.0f}  ({maskPixel.sum()} pixels)')


## Step 7: Apply count-threshold mask
Zeroes out pixels below `threshold` photon counts before the lifetime computation.

In [ ]:
bc = (slice(None), slice(None)) + (np.newaxis,) * (DataTCSPC.ndim - 2)
DataTCSPC = DataTCSPC * maskTCSPC[bc]
print(f'Pixels zeroed out: {(maskTCSPC == 0).sum():.0f}')

## Step 8: Mean-arrival-time lifetime (tav)
**Key equations:**
- `H = sum(tau * IRF) / sum(IRF)` — mean arrival time of the IRF (instrument offset)
- `F[x,y] = sum(tau * data[x,y]) / sum(data[x,y])` — mean photon arrival time per pixel
- `tav = F - H` — fluorescence lifetime estimate per pixel (ns)

In [ ]:
DataTCSPC = DataTCSPC.astype(float)
if DataTCSPC.ndim == 3:
    DataTCSPC = DataTCSPC[:, :, :, np.newaxis]

nx, ny, nbin, nch = DataTCSPC.shape
T = tau[np.newaxis, np.newaxis, :, np.newaxis]

H = float(np.dot(tau, IRF2) / IRF2.sum())

F = (T * DataTCSPC).sum(axis=2) / DataTCSPC.sum(axis=2)
F = np.nan_to_num(F)

tag = DataTCSPC.sum(axis=2).squeeze()
tav = (F - H).squeeze()
tav = np.clip(tav, 0, None)

print(f'H (IRF mean arrival time): {H:.4f} ns')
valid_tav = tav[maskTCSPCbool & (tav > 0)]
if valid_tav.size > 0:
    print(f'tav stats (masked) -- min: {valid_tav.min():.3f} ns,  '
          f'mean: {valid_tav.mean():.3f} ns,  max: {valid_tav.max():.3f} ns')

_tav_disp = tav.copy().astype(float)
_tav_disp[~maskTCSPCbool] = np.nan
_pos = _tav_disp[np.isfinite(_tav_disp) & (_tav_disp > 0)]
_vmin = float(np.percentile(_pos,  1)) if _pos.size else 0.0
_vmax = float(np.percentile(_pos, 90)) if _pos.size else 1.0

fig, ax = plt.subplots(figsize=(6, 5), facecolor='black')
ax.set_facecolor('black')
im = ax.imshow(_tav_disp, cmap='gnuplot2', aspect='equal', origin='upper',
               vmin=_vmin, vmax=_vmax)
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('tav / ns', color='white')
cbar.ax.yaxis.set_tick_params(color='white')
plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white')
cbar.outline.set_visible(False)
ax.set_title(f'Mean arrival-time lifetime  [{FName}]' + (f'\n{meta_str}' if meta_str else ''),
             fontsize=10, color='white')
ax.set_xlabel('x / px', color='white')
ax.set_ylabel('y / px', color='white')
ax.tick_params(colors='white')
for spine in ax.spines.values():
    spine.set_edgecolor('white')
plt.tight_layout()
plt.show()

## Step 9: Bilateral filter on tav
Edge-preserving smooth of the lifetime map. `sigma_spatial` controls neighbourhood size; `sigma_lifetime` (ns) controls how much lifetime difference is tolerated — sharp compartment boundaries are preserved while shot-noise within a compartment is averaged. Overwrites `tav` in place; skip to keep the raw map.

In [ ]:
# ── parameters ────────────────────────────────────────────────────────────────
sigma_spatial   = 4.0   # spatial Gaussian σ (pixels)   — larger = more smoothing
sigma_lifetime  = 0.4   # range σ (ns) — lifetimes within this are averaged;
                        #   increase to smooth across compartments, decrease to sharpen edges

# calc std of tav for later use in range-based filtering (optional)
tav_std = np.nanstd(tav[maskTCSPCbool & (tav > 0)])
print(f'tav std (masked, positive pixels): {tav_std:.3f} ns')
sigma_lifetime = max(sigma_lifetime, 0.5 * tav_std)
print(f'Using σ_lifetime = {sigma_lifetime:.3f} ns for bilateral filter')

# ── apply ─────────────────────────────────────────────────────────────────────
# Mask invalid pixels as NaN so they are excluded from their neighbours' averages.
_tav_in = tav.copy().astype(float)
_tav_in[~maskTCSPCbool] = np.nan

_tav_filled = np.where(np.isfinite(_tav_in), _tav_in, float(np.nanmean(_tav_in)))
_tav_filt   = denoise_bilateral(_tav_filled, sigma_color=sigma_lifetime,
                                sigma_spatial=sigma_spatial, channel_axis=None)
_tav_filt[~maskTCSPCbool] = np.nan

# ── percentile range over positive pixels only (zeros are clipped negatives) ──
_pos = _tav_in[np.isfinite(_tav_in) & (_tav_in > 0)]
_vmin = float(np.percentile(_pos,  20)) if _pos.size else 0.0
_vmax = float(np.percentile(_pos, 80)) if _pos.size else 1.0

# ── side-by-side comparison ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5), facecolor='black')
for ax in axes:
    ax.set_facecolor('black')

axes[0].imshow(_tav_in,   cmap='gnuplot2', aspect='equal', origin='upper', vmin=_vmin, vmax=_vmax)
axes[0].set_title('tav  (raw)', fontsize=11, color='white')
im = axes[1].imshow(_tav_filt, cmap='gnuplot2', aspect='equal', origin='upper', vmin=_vmin, vmax=_vmax)
axes[1].set_title(f'tav  (bilateral  σ_spatial={sigma_spatial} px, σ_range={sigma_lifetime} ns)',
                  fontsize=11, color='white')

cbar = plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
cbar.set_label('tav / ns', color='white')
cbar.ax.yaxis.set_tick_params(color='white')
plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white')
cbar.outline.set_visible(False)

for ax in axes:
    ax.tick_params(colors='white')
    for spine in ax.spines.values():
        spine.set_edgecolor('white')

_sup = FName + (f'  —  {meta_str}' if meta_str else '')
plt.suptitle(_sup, fontsize=9, y=1.01, color='white')
plt.tight_layout()
plt.show()

# overwrite tav so downstream cells (FLIM image, metrics) use the filtered map
tav = _tav_filt
print(f'tav updated  (bilateral filtered).  '
      f'Mean: {np.nanmean(tav):.3f} ns,  std: {np.nanstd(tav):.3f} ns')

## Step 10: False-colour FLIM image
Auto-determines lifetime display range with `uplowthresh` (if `limits` is NaN), then calls `FastFLIM_mod_AT0` to generate and display the false-colour image.

In [ ]:
saveLTimg = os.path.join(newFolder, FName)
# limits    = np.asarray(limits, dtype=float) # only needed if run on loop (get the first limits value, then reuse for consistent scaling across images)
limits    = np.array([np.nan, np.nan])
if np.all(np.isnan(limits)):
    ind = tag < threshold
    if not np.any(~ind):
        ind = tag < 1
    tim_auto = tav.copy()
    tim_auto[ind] = 0
    hi1, lo1 = uplowthresh(tim_auto, 0.95, 0.05)
    limits = np.array([lo1, hi1])
    print(f'Auto limits: [{limits[0]:.3f}, {limits[1]:.3f}] ns')

LTImg, Bar = FastFLIM_mod_AT0(saveLTimg, tag, tav, threshold, limits)

## Step 11: Phasor — z-projected, IRF corrected
Combines the full z-stack photon budget for a denser, lower-noise phasor:
1. **Z-projection** — accumulates all z-plane TCSPC arrays into one `(ny, nx, nbin)` image using the same `shift_data` alignment as the rest of the pipeline.
2. **G/S computation** — DFT at the laser repetition frequency with optional `szBin` spatial binning; no uncorrected phasor plot is shown.
3. **IRF reference correction** — maps the IRF phasor to `(G=1, S=0)`, applies the same modulation/phase factors to the sample. Results in `G_corr` / `S_corr`; corrected plot saved as `*_Phas_corr.png`.

`G_ref` / `S_ref` produced here are also consumed by the per-cell phasor cell (Step 15).

In [ ]:
freq_mhz = 1000.0 / Tcycle
tau_p    = (np.arange(1, nbin + 1) - 0.5) * dt

try:
    # ── IRF reference phasor ──────────────────────────────────────────────────
    _ph_irf = np.dot(tau_p, IRF2) / IRF2.sum()
    _T_irf  = 2.0 * np.pi * (tau_p - _ph_irf) / Tcycle
    _w_irf  = IRF2 / IRF2.sum()
    G_ref   = np.array([[float(np.dot(np.cos(_T_irf), _w_irf))]])
    S_ref   = np.array([[float(np.dot(np.sin(_T_irf), _w_irf))]])
    print(f'IRF phasor:  G_ref = {G_ref[0,0]:.4f},  S_ref = {S_ref[0,0]:.4f}')

    # ── accumulate z-projected TCSPC (one plane at a time) ───────────────────
    print(f'Accumulating z-projected TCSPC ({n_planes} planes)...', end=' ', flush=True)
    _tcspc_proj = np.zeros((nx, ny, nbin), dtype=float)
    for _path in data_paths:
        _DataZ = _to_yxt(tifffile.imread(_path).astype(float), dim_data)
        _tcspc_proj += np.roll(_DataZ, shift_data, axis=2)
    print('done.')

    # ── compute G / S without plot (mirrors Calc_Phasor_Mat internals) ───────
    _D = _tcspc_proj.copy()
    if szBin > 1:
        _nyb = (nx // szBin) * szBin
        _nxb = (ny // szBin) * szBin
        _D   = _D[:_nyb, :_nxb, :]
        _D   = _D.reshape(_nyb // szBin, szBin, _nxb // szBin, szBin, nbin).sum(axis=(1, 3))
    _tag   = _D.sum(axis=2)
    _ph    = np.dot(tau_p, IRF2) / IRF2.sum()
    _T     = 2.0 * np.pi * (tau_p - _ph) / Tcycle
    _T3    = _T[np.newaxis, np.newaxis, :]
    _tag_s = np.where(_tag > 0, _tag, np.nan)
    G      = (np.cos(_T3) * _D).sum(axis=2) / _tag_s
    S      = (np.sin(_T3) * _D).sum(axis=2) / _tag_s
    _bad   = (_tag < threshold) | (G < 0) | (S < 0)
    G[_bad] = np.nan;  S[_bad] = np.nan
    print(f'G raw : [{np.nanmin(G):.3f}, {np.nanmax(G):.3f}]  '
          f'S raw : [{np.nanmin(S):.3f}, {np.nanmax(S):.3f}]  '
          f'({int((~_bad).sum())} valid pixels after {szBin}x{szBin} binning)')

    # ── reference correction ──────────────────────────────────────────────────
    G_corr, S_corr, tau_mod, tau_phi = phasor_ref_correction(
        G, S, G_ref, S_ref, tau_ref_ns=0.0, freq_mhz=freq_mhz)
    print(f'G corr: [{np.nanmin(G_corr):.3f}, {np.nanmax(G_corr):.3f}]  '
          f'S corr: [{np.nanmin(S_corr):.3f}, {np.nanmax(S_corr):.3f}]')
    _vp = tau_phi[np.isfinite(tau_phi)] if tau_phi is not None else np.array([])
    if _vp.size:
        print(f'tau_phi: [{_vp.min():.3f}, {_vp.max():.3f}] ns')

    # ── build corrected phasor density matrix ─────────────────────────────────
    _gv = G_corr.ravel();  _sv = S_corr.ravel()
    _ok = np.isfinite(_gv) & np.isfinite(_sv)
    _gv, _sv = _gv[_ok], _sv[_ok]

    _nbin_p = int(round(1.0 / szPhs)) + 1
    _Xi     = np.round(_gv / szPhs).astype(int)
    _Yi     = np.round(_sv / szPhs).astype(int)
    M_corr  = np.zeros((_nbin_p, _nbin_p))
    _ok2    = (_Xi >= 0) & (_Xi < _nbin_p) & (_Yi >= 0) & (_Yi < _nbin_p)
    np.add.at(M_corr, (_Yi[_ok2], _Xi[_ok2]), 1)

    # ── corrected phasor plot ─────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(6, 6))
    fig.patch.set_facecolor('white');  ax.set_facecolor('white')
    if M_corr.max() > 0:
        _vmax_p = np.percentile(M_corr[M_corr > 0], 99)
        _cmap_p = plt.cm.jet.copy();  _cmap_p.set_under('white')
        ax.imshow(np.flipud(M_corr), extent=[0, 1, 0, 1], aspect='equal',
                  cmap=_cmap_p, origin='upper', vmin=1, vmax=_vmax_p)
    _theta = np.linspace(0, np.pi, 400)
    ax.plot(0.5 + 0.5 * np.cos(_theta), 0.5 * np.sin(_theta), '-', color='black', linewidth=1.5)
    _omega = 2.0 * np.pi * freq_mhz * 1e6
    for _tau_tick in [0.5, 1, 2, 4, 8]:
        _wt  = _omega * _tau_tick * 1e-9
        _g_t = 1.0 / (1.0 + _wt ** 2);  _s_t = _wt / (1.0 + _wt ** 2)
        ax.plot(_g_t, _s_t, 'o', color='black', markersize=5, zorder=5)
        ax.annotate(f'{_tau_tick} ns', xy=(_g_t, _s_t), color='black',
                    fontsize=8, xytext=(5, 3), textcoords='offset points')
    ax.set_xlim(0, 1.0);  ax.set_ylim(0, 0.8);  ax.set_aspect('equal')
    ax.set_xlabel('G  (corrected)', fontweight='bold', fontsize=14, fontstyle='italic')
    ax.set_ylabel('S  (corrected)', fontweight='bold', fontsize=14, fontstyle='italic')
    ax.tick_params(labelsize=11)
    ax.set_title(f'Phasor — 3D z-projected, IRF corrected  |  {_ok.sum()} pts  [{FName}]'
                 + (f'\n{meta_str}' if meta_str else ''), fontsize=10)
    plt.tight_layout()
    _fig_phas = os.path.join(newFolder, FName + '_Phas_corr.png')
    plt.savefig(_fig_phas, facecolor='white', bbox_inches='tight')
    plt.show()
    print(f'Saved: {_fig_phas}')

except Exception as _e:
    import traceback;  traceback.print_exc()
    _nbin3 = int(round(1.0 / szPhs)) + 1
    M_corr = np.zeros((_nbin3, _nbin3))
    G = S = G_corr = S_corr = tau_mod = tau_phi = G_ref = S_ref = None
    print(f'Phasor failed: {_e}')

## Step 12: Lifetime summary metrics
Three independent scalar estimates derived from `tav`:
- **Tau_av** (Method 1): mean of PMT-masked pixels where `tav > H/2`
- **Tau_norm** (Method 3): mean photon arrival time in the whole signal area minus H
- **Tau_mask** (Method 4): mean tav after re-applying the PMT mask to the photon data

In [ ]:
# Method 1 -- mean tav of PMT-masked pixels where tav > H/2
Tau_1  = tav * maskPMT
valid  = Tau_1[Tau_1 > (H / 2)]
Tau_av = float(np.nanmean(valid)) if valid.size > 0 else np.nan

# Method 3 -- mean photon arrival time across signal area minus H
Tau_norm = float(np.dot(tau, tmp1) / tmp1.sum() - H)

# Method 4 -- tav mean after re-masking photon data with PMT mask
DT_masked = DataTCSPC * maskPMT[:, :, np.newaxis, np.newaxis]
F1        = np.nansum(T * DT_masked, axis=2) / np.nansum(DT_masked, axis=2)
tav2      = np.clip((F1 - H).squeeze(), 0, None)
Tau_mask  = float(np.nanmean(tav2))

print(f'Tau_av   (Method 1 -- PMT mask, tav > H/2):  {Tau_av:.4f} ns')
print(f'Tau_norm (Method 3 -- area mean - H):         {Tau_norm:.4f} ns')
print(f'Tau_mask (Method 4 -- PMT-masked tav mean):   {Tau_mask:.4f} ns')

## Step 13: Z-stack loop (all planes)
Processes **all planes** in `data_paths` through the full pipeline.
`limits` (colourmap range) are fixed from the middle plane computed in Steps 1–10 above — run those cells first.

**Outputs per plane**
| File | Content |
|------|---------|
| `*_sumTCSPC_stack.tif` | Raw photon-count maps (float32, multi-page) |
| `*_sumTCSPC_filt_norm_stack.tif` | Filtered + normalised intensity (float32, multi-page) |
| `*_mask_stack.tif` | Threshold masks (float32, multi-page) |
| `*_tav_raw_stack.tif` | Per-pixel mean-arrival-time lifetime before bilateral filter |
| `*_tav_filt_stack.tif` | Bilateral-filtered lifetime (NaN = masked) |
| `*_FLIM_RGB_stack.tif` | False-colour RGB volume (uint8, multi-page) |
| `*_z###_LTimg.png` | Per-plane false-colour FLIM image |
| `*_z###_decay.png` | Per-plane aligned decay plot |
| `*_FLIM_limits.csv` | Colourmap limits (lo/hi ns) from middle plane |

In [ ]:
importlib.reload(_ffm_module)
from FastFLIM_mod_AT0 import FastFLIM_mod_AT0

# ── output stack paths ─────────────────────────────────────────────────────────
def _sp(tag): return os.path.join(newFolder, f'{FName}_{tag}.tif')
stack_sum      = _sp('sumTCSPC_stack')
stack_norm     = _sp('sumTCSPC_filt_norm_stack')
stack_mask     = _sp('mask_stack')
stack_tav_raw  = _sp('tav_raw_stack')
stack_tav_filt = _sp('tav_filt_stack')
stack_rgb      = _sp('FLIM_RGB_stack')

for _p in [stack_sum, stack_norm, stack_mask, stack_tav_raw, stack_tav_filt, stack_rgb]:
    if os.path.exists(_p): os.remove(_p)

# ── save colourmap limits to CSV ──────────────────────────────────────────────
csv_path = os.path.join(newFolder, f'{FName}_FLIM_limits.csv')
with open(csv_path, 'w', newline='') as _f:
    _w = csv.writer(_f)
    _w.writerow(['lo_ns', 'hi_ns', 'source_plane'])
    _w.writerow([f'{limits[0]:.4f}', f'{limits[1]:.4f}', f'z={mid_idx}'])
print(f'Limits  {limits[0]:.4f} – {limits[1]:.4f} ns  (from z={mid_idx})  →  {csv_path}')

# ── single global shift from bulk fluorescence ────────────────────────────────
# IRF2 was independently aligned to target by cell 5622bfa1 (using its own peak).
# Data is independently aligned to the same target using the bulk fluorescence peak.
# Both arrive at the same reference bin → F - H is unbiased, regardless of when IRF was acquired.
print(f'Computing bulk decay for global shift ({n_planes} planes)...', end=' ', flush=True)
_bulk = np.zeros(nbin, dtype=float)
for _p in data_paths:
    _bulk += _to_yxt(tifffile.imread(_p).astype(float), dim_data).sum(axis=(0, 1))
print('done.')

_peak_bulk = int(np.argmax(_bulk))
shift_bulk = target - _peak_bulk
H_bulk     = float(np.dot(tau, IRF2) / IRF2.sum())   # IRF2 already at target
T_4d       = tau[np.newaxis, np.newaxis, :, np.newaxis]

print(f'Bulk peak bin {_peak_bulk}  →  shift_bulk = {shift_bulk:+d}  |  H_bulk = {H_bulk:.4f} ns')
print(f'{n_planes} planes to process\n')

# ── create output subfolders ─────────────────────────────────────────────────
_zdecay_dir  = os.path.join(newFolder, 'z_decay_plots')
_flimrgb_dir = os.path.join(newFolder, 'FLIM_RGB_planes')
os.makedirs(_zdecay_dir,  exist_ok=True)
os.makedirs(_flimrgb_dir, exist_ok=True)

# ── open all TiffWriters once — keeps planes in the same series ───────────────
_tw_sum  = tifffile.TiffWriter(stack_sum)
_tw_norm = tifffile.TiffWriter(stack_norm)
_tw_mask = tifffile.TiffWriter(stack_mask)
_tw_traw = tifffile.TiffWriter(stack_tav_raw)
_tw_tflt = tifffile.TiffWriter(stack_tav_filt)
_tw_rgb  = tifffile.TiffWriter(stack_rgb)

try:
    # ══════════════════════════════════════════════════════════════════════════
    for z, path in enumerate(data_paths):

        # ── 1. Load plane ──────────────────────────────────────────────────────
        _raw  = tifffile.imread(path).astype(float)
        DataZ = _to_yxt(_raw, dim_data)
        sumZ  = DataZ.sum(axis=2)

        # ── 2. Mask from 3D pre-processed volume ──────────────────────────────
        maskZ_bool = mask3d[z]
        maskZ      = maskZ_bool.astype(float)
        sumZ_norm  = vol_norm_3d[z]

        # ── 3. Apply single global shift (same for all planes) ────────────────
        DataZ = np.roll(DataZ, shift_bulk, axis=2)

        # ── 4. Decay plot — saved, not displayed ──────────────────────────────
        _dec_n = DataZ[maskZ_bool].sum(axis=0).astype(float) if maskZ_bool.any() else DataZ.sum(axis=(0, 1))
        _dec_n = _dec_n / _dec_n.max()
        _irf_n = IRF2 / IRF2.max()
        _fig_d, _ax_d = plt.subplots(figsize=(6, 3.5))
        _ax_d.semilogy(tau, _irf_n, '--k', lw=0.8, label='IRF')
        _ax_d.semilogy(tau, _dec_n, color='steelblue', lw=1.2, label=f'z={z:03d}')
        _ax_d.axvline(tau[target], color='orange', lw=0.8, ls=':')
        _ax_d.set_xlabel('time / ns'); _ax_d.set_ylabel('norm. intensity')
        _ax_d.set_title(f'Aligned decay  [{FName}]  z={z:03d}/{n_planes-1}', fontsize=9)
        _ax_d.set_xlim(0, Tcycle); _ax_d.set_ylim(5e-3, 1.3); _ax_d.legend(fontsize=8)
        _fig_d.tight_layout()
        _fig_d.savefig(os.path.join(_zdecay_dir, f'{FName}_z{z:03d}_decay.png'),
                       dpi=100, bbox_inches='tight')
        plt.close(_fig_d)

        # ── 5. Write intensity / mask planes ──────────────────────────────────
        _tw_sum.write(sumZ.astype(np.float32), contiguous=True)
        _tw_norm.write(sumZ_norm.astype(np.float32), contiguous=True)
        _tw_mask.write(maskZ.astype(np.float32), contiguous=True)

        # ── 6. Apply mask and compute tav ─────────────────────────────────────
        DataZ = (DataZ * maskZ[:, :, np.newaxis]).astype(float)[:, :, :, np.newaxis]
        with np.errstate(invalid='ignore', divide='ignore'):
            _F_z = np.nan_to_num((T_4d * DataZ).sum(axis=2) / DataZ.sum(axis=2))
        tav_z = np.clip((_F_z - H_bulk).squeeze(), 0, None)
        _tw_traw.write(tav_z.astype(np.float32), contiguous=True)

        # ── 7. Bilateral filter ───────────────────────────────────────────────
        _tav_in = tav_z.copy(); _tav_in[~maskZ_bool] = np.nan
        _fill   = float(np.nanmean(_tav_in)) if np.any(np.isfinite(_tav_in)) else 0.0
        tav_filt_z = denoise_bilateral(
            np.where(np.isfinite(_tav_in), _tav_in, _fill),
            sigma_color=sigma_lifetime, sigma_spatial=sigma_spatial, channel_axis=None)
        tav_filt_z[~maskZ_bool] = np.nan
        _tw_tflt.write(tav_filt_z.astype(np.float32), contiguous=True)

        # ── 8. False-colour FLIM image ────────────────────────────────────────
        _png_base = os.path.join(_flimrgb_dir, f'{FName}_z{z:03d}')
        LTImg_z, _ = FastFLIM_mod_AT0(_png_base, sumZ_norm, tav_filt_z, thresh_norm, limits,
                                       show=(z == mid_idx))
        _tw_rgb.write((np.clip(LTImg_z, 0, 1) * 255).astype(np.uint8), photometric='rgb')

        # ── 9. Middle-plane QC visualisation ──────────────────────────────────
        if z == mid_idx:
            _pos_z = tav_filt_z[np.isfinite(tav_filt_z) & (tav_filt_z > 0)]
            _v0 = float(np.percentile(_pos_z,  1)) if _pos_z.size else 0.0
            _v1 = float(np.percentile(_pos_z, 99)) if _pos_z.size else 1.0
            _fig_m, _axes_m = plt.subplots(1, 2, figsize=(12, 5), facecolor='black')
            for _a in _axes_m: _a.set_facecolor('black')
            _axes_m[0].imshow(sumZ, cmap='inferno', aspect='equal', origin='upper')
            _axes_m[0].set_title(f'sumTCSPC  z={z}', color='white', fontsize=11)
            _im_m = _axes_m[1].imshow(tav_filt_z, cmap='jet', aspect='equal', origin='upper',
                                       vmin=_v0, vmax=_v1)
            _axes_m[1].set_title(f'tav filtered  z={z}', color='white', fontsize=11)
            _cb_m = plt.colorbar(_im_m, ax=_axes_m[1], fraction=0.046, pad=0.04)
            _cb_m.set_label('tav / ns', color='white')
            _cb_m.ax.yaxis.set_tick_params(color='white')
            plt.setp(_cb_m.ax.yaxis.get_ticklabels(), color='white')
            _cb_m.outline.set_visible(False)
            for _a in _axes_m:
                _a.tick_params(colors='white')
                for _s in _a.spines.values(): _s.set_edgecolor('white')
            plt.suptitle(f'{FName}  —  middle plane z={z}  (loop QC)', color='white', fontsize=9)
            plt.tight_layout()
            plt.show()

        print(f'  z={z:03d}/{n_planes-1}  tav={np.nanmean(tav_filt_z):.3f} ns  {os.path.basename(path)}')
    # ══════════════════════════════════════════════════════════════════════════

finally:
    for _tw in [_tw_sum, _tw_norm, _tw_mask, _tw_traw, _tw_tflt, _tw_rgb]:
        _tw.close()

print(f'\nDone.  {n_planes} planes processed.')
print(f'Output folder:  {newFolder}')

## Step 14: Per-cell 3D decay fitting
Fits a multi-exponential decay to each Cellpose cell's pooled 3D TCSPC.

**Speed-ups vs original:**
- **Single-pass I/O**: bulk decay and per-cell accumulation are done in one loop over z-planes (was two).
- **Vectorised accumulation**: one-hot matmul + `bincount` replace the inner `for cid in _cids` Python loop.
- **Parallel fitting**: `ThreadPoolExecutor` runs `Fit_decay` concurrently across cells.
- **Two-tier cache**:
  - `*_cell_acc.npz` — accumulated per-cell TCSPC + centroid arrays. Delete to force re-accumulation.
  - `*_per_cell_lifetimes.csv` — fitting results. Delete (keep `.npz`) to re-fit without re-accumulating.

`_cell_decays` dict is always reconstructed after loading/computing and is available for Step 15.

In [ ]:
import time as _time
importlib.reload(_ffm_module)
import flim_helpers as _fh_mod; importlib.reload(_fh_mod)
from flim_helpers import Fit_decay
from scipy.signal import find_peaks

# ── parameters ───────────────────────────────────────────────────────────────
min_photons = 500

# ── cache paths ──────────────────────────────────────────────────────────────
_cache_acc  = os.path.join(newFolder, f'{FName}_cell_acc.npz')
csv_out     = os.path.join(newFolder, f'{FName}_per_cell_lifetimes.csv')
_fits_path  = os.path.join(newFolder, f'{FName}_cell_fits.npz')   # decay + model per cell
_have_acc   = os.path.exists(_cache_acc)
_have_csv   = os.path.exists(csv_out)
_have_fits  = os.path.exists(_fits_path)

# ════════════════════════════════════════════════════════════════════════════
# STAGE 1 — per-cell TCSPC accumulation  (cached in _cache_acc)
#   Delete _cache_acc to force re-accumulation.
# ════════════════════════════════════════════════════════════════════════════
if _have_acc:
    print(f'Loading accumulation cache: {os.path.basename(_cache_acc)}')
    _acc        = np.load(_cache_acc)
    bulk_decay  = _acc['bulk_decay']
    _cell_mat   = _acc['cell_decays']
    _cell_vox   = _acc['cell_voxels']
    _cell_z_arr = _acc['cell_z_acc']
    _cell_y_arr = _acc['cell_y_acc']
    _cell_x_arr = _acc['cell_x_acc']
    print(f'Loaded {n_cells} cells,  {int(_cell_vox[1:].sum()):,} voxels total.')
else:
    _ny_p, _nx_p = cp_masks.shape[1], cp_masks.shape[2]
    _ys_flat = np.repeat(np.arange(_ny_p, dtype=float), _nx_p)
    _xs_flat = np.tile  (np.arange(_nx_p, dtype=float), _ny_p)
    _eye_oh  = np.eye(n_cells + 1, dtype=np.float32)

    bulk_decay  = np.zeros(nbin,               dtype=float)
    _cell_mat   = np.zeros((n_cells + 1, nbin), dtype=float)
    _cell_vox   = np.zeros(n_cells + 1,        dtype=float)
    _cell_z_arr = np.zeros(n_cells + 1,        dtype=float)
    _cell_y_arr = np.zeros(n_cells + 1,        dtype=float)
    _cell_x_arr = np.zeros(n_cells + 1,        dtype=float)

    print(f'Accumulating {n_planes} planes (vectorised, single pass)...')
    _t0 = _time.perf_counter()
    for _z, _path in enumerate(data_paths):
        _DataZ     = _to_yxt(tifffile.imread(_path).astype(float), dim_data)
        _mask_flat = cp_masks[_z].ravel()
        _data_flat = _DataZ.reshape(-1, nbin)
        bulk_decay += _data_flat.sum(axis=0)
        _cell_mat  += _eye_oh[_mask_flat].T @ _data_flat
        _bc          = np.bincount(_mask_flat, minlength=n_cells + 1).astype(float)
        _cell_vox   += _bc
        _cell_z_arr += _bc * _z
        _cell_y_arr += np.bincount(_mask_flat, weights=_ys_flat, minlength=n_cells + 1)
        _cell_x_arr += np.bincount(_mask_flat, weights=_xs_flat, minlength=n_cells + 1)
        if (_z + 1) % max(1, n_planes // 10) == 0 or _z == n_planes - 1:
            print(f'  plane {_z + 1}/{n_planes}', end='\r', flush=True)

    print(f'\nDone in {_time.perf_counter() - _t0:.1f} s.')
    np.savez_compressed(_cache_acc,
                        bulk_decay=bulk_decay,  cell_decays=_cell_mat,
                        cell_voxels=_cell_vox,  cell_z_acc=_cell_z_arr,
                        cell_y_acc=_cell_y_arr, cell_x_acc=_cell_x_arr)
    print(f'Saved accumulation cache → {os.path.basename(_cache_acc)}')

_cell_decays = {cid: _cell_mat[cid] for cid in range(1, n_cells + 1)}

# ── Shift bulk decay + all cell decays to 8% target (consistent with Step 5) ──
_peak_bulk       = int(np.argmax(bulk_decay))
_peak_irf        = int(np.argmax(IRF2))
_shift_to_target = target - _peak_bulk          # bins to move peak to target
bulk_decay = np.roll(bulk_decay, _shift_to_target)
_cell_mat  = np.roll(_cell_mat,  _shift_to_target, axis=1)
IRF2_fit   = np.roll(IRF2, target - _peak_irf)  # IRF also goes to target
print(f'Bulk peak bin {_peak_bulk}  →  shifted {_shift_to_target:+d} bins  →  target bin {target}  |  IRF shifted to target')

# ── Auto-detect coherent artifact bump ────────────────────────────────────────
# Tweak these if the detected bump is wrong (check the shaded region on the plot).
bump_srch_lo_ns    = 3.5   # ← start of search window, ns after peak; raise if still in decay tail
bump_srch_hi_ns    = 7.0   # ← end of search window
bump_safety_bins   = 4     # ← bins to cut before bump centre → t_stop
bump_min_prominence = 0.02  # ← min prominence as fraction of window peak (lower = more sensitive)
_srch_lo = min(nbin - 1, target + max(5, int(round(bump_srch_lo_ns / dt))))
_srch_hi = min(nbin,     target + int(round(bump_srch_hi_ns / dt)))
if _srch_hi > _srch_lo + 3:
    _smooth  = np.convolve(bulk_decay, np.ones(5) / 5, mode='same')
    _win     = _smooth[_srch_lo:_srch_hi]
    _win_n   = _win / _win.max() if _win.max() > 0 else _win
    _peaks, _props = find_peaks(_win_n, prominence=bump_min_prominence)
    print(f'Search window: {tau[_srch_lo]:.2f} – {tau[min(_srch_hi-1,nbin-1)]:.2f} ns  '
          f'(bins {_srch_lo}–{_srch_hi})  |  {len(_peaks)} peak(s) found')
    if _peaks.size:
        _best_rel   = _peaks[np.argmax(_win_n[_peaks])]
        _bump_bin   = _srch_lo + int(_best_rel)
        _bump_ns    = float(tau[_bump_bin])
        _t_stop_bin = max(0, _bump_bin - bump_safety_bins)
        t_stop_ns   = float(tau[_t_stop_bin])
        print(f'Coherent bump: bin {_bump_bin}  ({_bump_ns:.3f} ns)  '
              f'→  t_stop = {t_stop_ns:.3f} ns  (bin {_t_stop_bin})')
    else:
        _bump_bin   = _srch_lo
        _bump_ns    = float(tau[_bump_bin])
        _t_stop_bin = max(0, _bump_bin - bump_safety_bins)
        t_stop_ns   = float(tau[_t_stop_bin])
        print(f'WARNING: no peak found (prominence >= {bump_min_prominence}) in bump search window — '
              f'defaulting bump to window start ({_bump_ns:.3f} ns). '
              f'Lower bump_min_prominence or adjust bump_srch_lo/hi_ns.')
        print(f'  t_stop = {t_stop_ns:.3f} ns  (bin {_t_stop_bin})')
else:
    t_stop_ns = None;  _bump_ns = None
    print('Bump search window too narrow — fitting full decay.')

# ── QC: bulk decay plot ───────────────────────────────────────────────────────
_bulk_n = bulk_decay / bulk_decay.max()
_irf_n  = IRF2_fit  / IRF2_fit.max()
fig_bulk, ax_bulk = plt.subplots(figsize=(7, 4))
ax_bulk.semilogy(tau, _irf_n,  '--k', lw=0.8, label='IRF (aligned)')
ax_bulk.semilogy(tau, _bulk_n, color='steelblue', lw=1.2, label='Bulk 3D decay')
ax_bulk.axvline(tau[target], color='orange', lw=0.8, ls=':', label=f'peak  bin {target}  (8% target)')
if t_stop_ns is not None:
    ax_bulk.axvspan(tau[_srch_lo], tau[min(_srch_hi - 1, nbin - 1)],
                    alpha=0.08, color='red', label='search window')
    ax_bulk.axvline(_bump_ns,  color='orangered', lw=0.8, ls=':', label=f'bump  {_bump_ns:.2f} ns')
    ax_bulk.axvline(t_stop_ns, color='red',       lw=1.0, ls='--', label=f't_stop  {t_stop_ns:.2f} ns')
ax_bulk.set_xlabel('time / ns');  ax_bulk.set_ylabel('intensity (norm.)')
ax_bulk.set_title(f'Bulk 3D decay  [{FName}]', fontsize=10)
ax_bulk.set_xlim(0, Tcycle);  ax_bulk.set_ylim(5e-3, 1.3);  ax_bulk.legend(fontsize=8)
fig_bulk.tight_layout()
plt.show()

# ════════════════════════════════════════════════════════════════════════════
# STAGE 2 — per-cell Fit_decay  (sequential; cached in csv_out + _fits_path)
#   Delete csv_out (and _fits_path) to re-fit without re-accumulating.
# ════════════════════════════════════════════════════════════════════════════
if _have_acc and _have_csv and _have_fits:
    print(f'Reloading fitting results: {os.path.basename(csv_out)}')
    df_cells = pd.read_csv(csv_out)
    _fits     = np.load(_fits_path)
    _cell_models = _fits['models']   # (n_cells, nbin)
    print(df_cells[['cell_id', 'n_voxels', 'n_photons', 'aw_LT', 'chi2']].to_string(index=False))
else:
    records      = []
    _cell_models = np.full((n_cells, nbin), np.nan)   # row i = cell i+1

    _t_stop_str = f't_stop={t_stop_ns:.3f} ns' if t_stop_ns else 't_stop=None'
    print(f'Fitting {n_cells} cells  (min_photons={min_photons},  {_t_stop_str})...')
    _t_fit = _time.perf_counter()

    for cid in range(1, n_cells + 1):
        decay = _cell_mat[cid]
        n_vox = int(_cell_vox[cid])
        n_ph  = int(decay.sum())
        cz = float(_cell_z_arr[cid] / n_vox) if n_vox else np.nan
        cy = float(_cell_y_arr[cid] / n_vox) if n_vox else np.nan
        cx = float(_cell_x_arr[cid] / n_vox) if n_vox else np.nan
        rec = dict(cell_id=cid, n_voxels=n_vox, n_photons=n_ph,
                   centroid_z=round(cz, 2), centroid_y=round(cy, 2),
                   centroid_x=round(cx, 2))
        if n_ph < min_photons:
            rec.update(aw_LT=np.nan, chi2=np.nan)
            for i in range(1, 6):
                rec[f'tau{i}'] = np.nan;  rec[f'amp{i}'] = np.nan
            print(f'  cell {cid:3d}  SKIP  (n_ph={n_ph})')
        else:
            print(f'  cell {cid:3d}  fitting...', end='\r', flush=True)
            _t2, _a2, _, _model_curve, _chi2, _ = Fit_decay(
                IRF2_fit, decay, Tcycle, dt, t_stop_ns=t_stop_ns)
            _cell_models[cid - 1] = _model_curve
            _t2a = np.asarray(_t2).ravel();  _a2a = np.asarray(_a2).ravel()
            _valid = _a2a > 0
            _aw = (float(np.dot(_t2a[_valid], _a2a[_valid]) / _a2a[_valid].sum())
                   if _valid.any() else np.nan)
            rec.update(aw_LT=round(_aw, 4), chi2=round(float(_chi2), 4))
            for i in range(5):
                rec[f'tau{i+1}'] = round(float(_t2a[i]), 4) if i < len(_t2a) else np.nan
                rec[f'amp{i+1}'] = round(float(_a2a[i]), 4) if i < len(_a2a) else np.nan
            print(f'  cell {cid:3d}  vox={n_vox:5d}  ph={n_ph:8,d}  '
                  f'aw_LT={_aw:.3f} ns  chi2={_chi2:.3f}')
        records.append(rec)

    print(f'Fitting done in {_time.perf_counter() - _t_fit:.1f} s.')
    df_cells = pd.DataFrame(records)
    df_cells.to_csv(csv_out, index=False)
    np.savez_compressed(_fits_path,
                        tau_axis    = tau,
                        irf_aligned = IRF2_fit,
                        t_stop_ns   = np.array([t_stop_ns if t_stop_ns else np.nan]),
                        cell_ids    = np.arange(1, n_cells + 1),
                        decays      = _cell_mat[1:],    # (n_cells, nbin) raw TCSPC
                        models      = _cell_models)     # (n_cells, nbin) fitted model
    print(f'Saved: {csv_out}')
    print(f'Saved: {os.path.basename(_fits_path)}  '
          f'(keys: tau_axis, irf_aligned, t_stop_ns, cell_ids, decays, models)')
    print(df_cells[['cell_id', 'n_voxels', 'n_photons', 'aw_LT', 'chi2']].to_string(index=False))

In [ ]:
# ── Cell to inspect: change this number ──────────────────────────────────────
CELL_NUM = 1          # 1-based cell ID (1 ... n_cells)

# ── Reload flim_helpers so any edits to the module take effect ───────────
import flim_helpers as _fh_mod; importlib.reload(_fh_mod)
from flim_helpers import Fit_decay

# ── Load saved decays + models from disk ───────────────────────────
if not os.path.exists(_fits_path):
    raise FileNotFoundError(
        f"Fits file not found -- run the per-cell fitting cell first.\n{_fits_path}")

_fits       = np.load(_fits_path, allow_pickle=True)
_t          = _fits["tau_axis"]         # (nbin,) time axis in ns
_irf        = _fits["irf_aligned"]      # (nbin,) aligned IRF
_all_decays = _fits["decays"]           # (n_cells, nbin) raw TCSPC
_all_models = _fits["models"]           # (n_cells, nbin) fitted model
_t_stop_val = _fits["t_stop_ns"]
_t_stop_item = _t_stop_val.item()
_t_stop      = float(_t_stop_item) if (_t_stop_item is not None and np.isfinite(float(_t_stop_item))) else None

# ── Reload per-cell CSV for metadata ──────────────────────────────
df_cells = pd.read_csv(csv_out)

# ── Validate requested cell ────────────────────────────────────────
_nc = _all_decays.shape[0]
if not (1 <= CELL_NUM <= _nc):
    raise ValueError(f"CELL_NUM={CELL_NUM} out of range -- dataset has {_nc} cells (1-{_nc})")

_raw = _all_decays[CELL_NUM - 1].astype(float)
_mod = _all_models[CELL_NUM - 1].astype(float)
_row = df_cells[df_cells["cell_id"] == CELL_NUM].iloc[0]

# ── Normalise to peak of raw decay ─────────────────────────────────
_pk    = _raw.max() if _raw.max() > 0 else 1.0
_raw_n = _raw / _pk
_mod_n = _mod / _pk
_irf_n = _irf / _irf.max()

# ── Plot ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(_t, _raw_n, s=8,  color="steelblue", alpha=0.75, zorder=2, label="raw data")
ax.plot   (_t, _mod_n, "-",  color="tomato",    linewidth=1.8, zorder=3, label="model fit")
ax.plot   (_t, _irf_n, "--", color="0.45",      linewidth=0.8,           label="IRF (aligned)")
if _t_stop is not None:
    ax.axvline(_t_stop, color="red", linestyle=":", linewidth=1.0,
               label=f"t_stop = {_t_stop:.2f} ns")

ax.set_yscale("log")
ax.set_xlim(_t[0], _t[-1])
ax.set_ylim(5e-3, 1.5)
ax.set_xlabel("time / ns")
ax.set_ylabel("intensity  (norm. to peak)")
ax.set_title(
    f"{FName}  --  cell {CELL_NUM}  |  "
    f"aw_LT = {_row['aw_LT']:.3f} ns   "
    f"chi2 = {_row['chi2']:.2f}   "
    f"n_photons = {int(_row['n_photons']):,}"
)
ax.legend(fontsize=9, framealpha=0.6)
plt.tight_layout()
plt.show()


## Step 15: Per-cell phasor (Cellpose-masked, IRF corrected)
Computes a single corrected (G, S) phasor coordinate per Cellpose cell from its pooled 3D decay.
Requires `G_ref` / `S_ref` from Step 11 and `_cell_decays` from Step 14.

Two scatter panels on the universal semicircle:
- **Left** — dots colored by amplitude-weighted lifetime (`aw_LT`)
- **Right** — dots colored by centroid depth (`centroid_z`)

Dot labels show `cell_id`. Back-projection: use `cp_masks == cell_id` to locate those voxels in 3D.
`G_phasor` / `S_phasor` are appended to the per-cell CSV.

In [ ]:
if G_ref is None or S_ref is None:
    print('G_ref / S_ref not available — run Step 13 first.')
elif not _cell_decays:
    print('_cell_decays is empty — run the per-cell fitting cell first.')
else:
    _tau_p = (np.arange(1, nbin + 1) - 0.5) * dt
    _ph_c  = np.dot(_tau_p, IRF2) / IRF2.sum()   # same IRF phase reference as Step 13
    _T_c   = 2.0 * np.pi * (_tau_p - _ph_c) / Tcycle
    _cos_T = np.cos(_T_c)
    _sin_T = np.sin(_T_c)

    # ── per-cell G / S ────────────────────────────────────────────────────────
    _G_raw = np.full(n_cells, np.nan)
    _S_raw = np.full(n_cells, np.nan)
    for _i, _cid in enumerate(range(1, n_cells + 1)):
        _dec = _cell_decays[_cid]
        _n   = float(_dec.sum())
        if _n >= min_photons:
            _w = _dec / _n
            _G_raw[_i] = float(np.dot(_cos_T, _w))
            _S_raw[_i] = float(np.dot(_sin_T, _w))

    # ── reference correction (same M_cor / phi_cor as Step 13) ───────────────
    _G_corr_c, _S_corr_c, _, _ = phasor_ref_correction(
        _G_raw, _S_raw, G_ref, S_ref, tau_ref_ns=0.0, freq_mhz=freq_mhz)

    # ── append to df_cells and re-save CSV ────────────────────────────────────
    df_cells['G_phasor'] = [round(float(g), 5) if np.isfinite(g) else np.nan
                            for g in _G_corr_c]
    df_cells['S_phasor'] = [round(float(s), 5) if np.isfinite(s) else np.nan
                            for s in _S_corr_c]
    df_cells.to_csv(csv_out, index=False)
    print(f'Added G_phasor / S_phasor  →  {os.path.basename(csv_out)}')

    # ── scatter plot: two panels ──────────────────────────────────────────────
    _ok_c   = np.isfinite(_G_corr_c) & np.isfinite(_S_corr_c)
    _g_plot = _G_corr_c[_ok_c]
    _s_plot = _S_corr_c[_ok_c]
    _ids    = df_cells['cell_id'].values[_ok_c]
    _aw     = df_cells['aw_LT'].values[_ok_c].astype(float)
    _cz     = df_cells['centroid_z'].values[_ok_c].astype(float)

    _aw_lo, _aw_hi = np.nanpercentile(_aw, [5, 95])  # clamp colormap to 5-95 pct
    _vlims = [(_aw_lo, _aw_hi), (None, None)]        # (vmin, vmax) per panel

    _theta = np.linspace(0, np.pi, 400)
    _omega = 2.0 * np.pi * freq_mhz * 1e6

    fig_cp, axes_cp = plt.subplots(1, 2, figsize=(13, 6))
    for _ax, _c, _lbl, _cm, (_vmin, _vmax) in zip(
            axes_cp,
            [_aw, _cz],
            ['aw_LT (ns)', f'centroid z  (0 – {n_planes - 1})'],
            ['plasma', 'viridis'],
            _vlims):
        _ax.plot(0.5 + 0.5 * np.cos(_theta), 0.5 * np.sin(_theta), '-k', linewidth=1.5)
        _sc = _ax.scatter(_g_plot, _s_plot, c=_c, cmap=_cm, s=60,
                          edgecolors='black', linewidths=0.5, zorder=5,
                          vmin=_vmin, vmax=_vmax)
        plt.colorbar(_sc, ax=_ax, fraction=0.046, pad=0.04).set_label(_lbl)
        for _t in [0.5, 1, 2, 4, 8]:
            _wt  = _omega * _t * 1e-9
            _g_t = 1.0 / (1.0 + _wt ** 2);  _s_t = _wt / (1.0 + _wt ** 2)
            _ax.plot(_g_t, _s_t, 'o', color='gray', markersize=4, zorder=4)
            _ax.annotate(f'{_t} ns', xy=(_g_t, _s_t), fontsize=7, color='gray',
                         xytext=(4, 2), textcoords='offset points')
        for _cid_lbl, _gv, _sv in zip(_ids, _g_plot, _s_plot):
            _ax.annotate(str(int(_cid_lbl)), xy=(_gv, _sv), fontsize=6,
                         ha='center', va='bottom', xytext=(0, 4), textcoords='offset points')
        _ax.set_xlim(0, 1.0);  _ax.set_ylim(0, 0.8);  _ax.set_aspect('equal')
        _ax.set_xlabel('G  (corrected)', fontweight='bold', fontsize=12, fontstyle='italic')
        _ax.set_ylabel('S  (corrected)', fontweight='bold', fontsize=12, fontstyle='italic')
    axes_cp[0].set_title(f'Per-cell phasor — color: aw_LT  [{FName}]', fontsize=10)
    axes_cp[1].set_title(f'Per-cell phasor — color: centroid z  [{FName}]', fontsize=10)
    plt.suptitle(f'{int(_ok_c.sum())} cells with ≥ {min_photons} photons', fontsize=9, y=1.01)
    plt.tight_layout()
    _fig_cp_phas = os.path.join(newFolder, FName + '_per_cell_phasor.png')
    plt.savefig(_fig_cp_phas, facecolor='white', bbox_inches='tight')
    plt.savefig(_fig_cp_phas.replace('.png', '.svg'), facecolor='white', bbox_inches='tight')
    plt.show()
    print(f'Saved: {_fig_cp_phas}  +  .svg')

## Step 16: Per-cell tav statistics (Cellpose-masked, bilateral-filtered)
Computes per-cell lifetime statistics from the bilateral-filtered `tav_filt_stack.tif` produced by Step 13.

For each Cellpose cell, **valid pixels** are those that are finite and positive within the cell mask.
Reported per cell: `n_valid_px`, `mean_tav`, `std_tav`, `median_tav`, `p25_tav`, `p75_tav`, `iqr_tav`.

Saved to `*_per_cell_tav.csv`. Run Step 13 first.

In [ ]:
# ── output path ──────────────────────────────────────────────────────────────
_tav_csv_out = os.path.join(newFolder, f'{FName}_per_cell_tav.csv')

if not os.path.exists(stack_tav_filt):
    print(f'ERROR: {os.path.basename(stack_tav_filt)} not found — run Step 13 first.')
else:
    # ── load full filtered-tav stack ──────────────────────────────────────────
    # tifffile writes each plane as a separate series when contiguous=True is
    # omitted — imread then returns only the first page.  Read all pages explicitly.
    with tifffile.TiffFile(stack_tav_filt) as _tf:
        _tav_stack = np.stack([p.asarray() for p in _tf.pages]).astype(float)
    if _tav_stack.ndim == 2:
        _tav_stack = _tav_stack[np.newaxis]
    _nz = _tav_stack.shape[0]
    print(f'Loaded tav_filt_stack: {_nz} planes, shape {_tav_stack.shape}')

    # ── accumulate per-cell tav pixel lists ──────────────────────────────────
    _cell_vals = {cid: [] for cid in range(1, n_cells + 1)}

    for _z in range(_nz):
        _tav_z  = _tav_stack[_z]                         # (Y, X)
        _mask_z = cp_masks[_z]                           # (Y, X) int cell IDs
        _valid  = np.isfinite(_tav_z) & (_tav_z > 0)
        _cids_v = _mask_z[_valid]
        _tav_v  = _tav_z[_valid]
        # group by cell ID without an inner Python loop
        _sort_i = np.argsort(_cids_v, kind='stable')
        _cids_s = _cids_v[_sort_i]
        _tav_s  = _tav_v[_sort_i]
        _uniq, _cnts = np.unique(_cids_s, return_counts=True)
        for _cid, _chunk in zip(_uniq, np.split(_tav_s, np.cumsum(_cnts)[:-1])):
            if 1 <= _cid <= n_cells:
                _cell_vals[int(_cid)].append(_chunk)

    # ── compute statistics ────────────────────────────────────────────────────
    _records = []
    for cid in range(1, n_cells + 1):
        _chunks = _cell_vals[cid]
        if _chunks:
            _arr = np.concatenate(_chunks)
            _p25, _p75 = np.percentile(_arr, [25, 75])
            _records.append(dict(
                cell_id    = cid,
                n_valid_px = len(_arr),
                mean_tav   = round(float(np.mean(_arr)),   4),
                std_tav    = round(float(np.std(_arr)),    4),
                median_tav = round(float(np.median(_arr)), 4),
                p25_tav    = round(float(_p25),            4),
                p75_tav    = round(float(_p75),            4),
                iqr_tav    = round(float(_p75 - _p25),     4),
            ))
        else:
            _records.append(dict(
                cell_id=cid, n_valid_px=0,
                mean_tav=np.nan, std_tav=np.nan, median_tav=np.nan,
                p25_tav=np.nan, p75_tav=np.nan, iqr_tav=np.nan,
            ))

    df_tav = pd.DataFrame(_records)
    df_tav.to_csv(_tav_csv_out, index=False)

    _n_ok = (df_tav.n_valid_px > 0).sum()
    print(f'Saved → {_tav_csv_out}')
    print(f'{_n_ok} / {n_cells} cells have valid tav pixels.')
    print(df_tav[['cell_id', 'n_valid_px', 'mean_tav', 'std_tav', 'median_tav', 'iqr_tav']].to_string(index=False))


---
## Retired / Reference cells
The cells below are **not part of the standard analysis** — they are kept as reference only.
They are all-commented or reference variables from disabled steps.
Do not run them in a fresh session without adapting the variable names.

### [Retired] Bin alignment
Not needed for the current instrument (`nbin_data == Ix`). Re-enable if IRF and data have mismatched bin counts.

In [ ]:
# ── Bin alignment -- not needed for new instrument (nbin_data == Ix assumed) ──
# nbin_data = DataTCSPC.shape[2]
# nbin      = min(nbin_data, Ix)
#
# if nbin < Ix:
#     nrep = int(Ix // nbin)
#     IRF2 = IRF2[:nbin * nrep].reshape(nrep, nbin).mean(axis=0)
#     print(f'IRF averaged: {Ix} -> {nbin} bins  (nrep={nrep})')
# else:
#     nbin   = Ix
#     nx_d, ny_d = DataTCSPC.shape[0], DataTCSPC.shape[1]
#     nrep   = int(nbin_data // nbin)
#     DataTCSPC = DataTCSPC[:, :, :nbin * nrep]
#     DataTCSPC = DataTCSPC.reshape(nx_d, ny_d, nrep, nbin).swapaxes(2, 3)
#     print(f'Data reshaped: {nbin_data} -> ({nbin} bins x {nrep} reps)')
#
# print(f'nbin = {nbin},  DataTCSPC shape = {DataTCSPC.shape}')

### [Retired] Manual IRF peak alignment
Replaced by the peak realignment cell (Step 5). Kept as reference for the alignment logic.

In [ ]:
# ── IRF-to-data peak alignment (replaced by realignment cell -- kept for reference)
# xs1 = int(np.argmax(irf))
# xs2 = int(np.argmax(tmp))
# if xs1 > xs2:
#     shift = xs1 - xs2
#     irf   = np.concatenate([irf[shift:],   np.zeros(shift)])
#     IRF2  = np.concatenate([IRF2[shift:],  np.zeros(shift)])
#     print(f'IRF shifted left by {shift} bins')
# elif xs1 < xs2:
#     shift = xs2 - xs1
#     irf   = np.concatenate([np.zeros(shift), irf[:-shift]])
#     IRF2  = np.concatenate([np.zeros(shift), IRF2[:-shift]])
#     print(f'IRF shifted right by {shift} bins')
# else:
#     print('Peaks already aligned')
#
# fig_lt, ax_lt = plt.subplots(figsize=(7, 4))
# ax_lt.semilogy(tau, irf, '--k', linewidth=0.5, label='IRF')
# ax_lt.semilogy(tau, tmp, ':o', markersize=4, markerfacecolor='b', color='b', label='Data')
# ax_lt.set_ylabel('signal / a.u.')
# ax_lt.set_xlabel('time / ns')
# ax_lt.set_xlim(0, np.ceil(Tcycle))
# ax_lt.set_ylim(5e-3, 1.2)
# ax_lt.legend()
# fig_lt.suptitle(f'LT Data & IRF  {FName}')
# plt.tight_layout()
# plt.show()

### [Retired] Single-plane Fit_decay on middle-plane decay
Fits `Fit_decay` on the spatially-averaged middle-plane decay `tmp1`. Superseded by the per-cell 3D fitting (Step 14). Re-enable and adapt variable names if a single-plane bulk fit is needed.

In [ ]:
# tau2, amp2, off2, _, chi2_2, model2 = Fit_decay(IRF2, tmp1, Tcycle, dt)
#
# # amplitude-weighted lifetime
# valid_mask = amp2 > 0
# aw_LT2 = (float(np.dot(tau2[valid_mask], amp2[valid_mask]) / amp2[valid_mask].sum())
#           if valid_mask.any() and amp2[valid_mask].sum() != 0 else np.nan)
#
# # cap to 5 components for storage
# tau2     = np.asarray(tau2).ravel()[:5]
# amp2     = np.asarray(amp2).ravel()[:5]
# tau_tmp2 = np.zeros(5); tau_tmp2[:len(tau2)] = tau2
# amp_tmp2 = np.zeros(5); amp_tmp2[:len(amp2)] = amp2
#
# print(f'Fit_decay:  chi2 = {chi2_2:.4f},  aw_LT = {aw_LT2:.4f} ns')
# for i, (t_, a_) in enumerate(zip(tau2, amp2)):
#     if a_ > 0:
#         print(f'  Component {i+1}: tau = {t_:.4f} ns,  amp = {a_:.4f}')

### [Retired] Save and collect results
Collects per-session lifetime scalars and saves the middle-plane LT figure. References `fig_lt`, `model2`, `tau_tmp2`, `amp_tmp2`, `aw_LT2`, `chi2_2` — all from the retired steps above.

In [ ]:
# fig_name1 = os.path.join(newFolder, FName + '_LT.png')
# fig_lt.savefig(fig_name1)
# print(f'LT figure saved: {fig_name1}')
#
# Dcays1   = {'ns': tau, 'irf_counts': IRF2}
# Dcays2   = {'tmp1': tmp1, 'model': model2}
# lifetime = {
#     'Tau_av':   Tau_av,
#     'Tau_mask': Tau_mask,
#     'Tau_norm': Tau_norm,
#     'tau_tmp2': tau_tmp2,
#     'amp_tmp2': amp_tmp2,
#     'aw_LT2':   aw_LT2,
#     'chi2':     chi2_2,
# }
# print()
# print('=== Lifetime summary ===')
# for k, v in lifetime.items():
#     print(f'  {k:12s}: {v}')